Notebook for ingesting FIFA World Cup Data

In [0]:
# ============================================================
# CELL 0 — Config and Setup
# ============================================================
import kagglehub
from kagglehub import KaggleDatasetAdapter
from pyspark.sql import functions as F

catalog = dbutils.widgets.get("catalog") if "catalog" in dbutils.widgets.getAll() else "main"
schema  = dbutils.widgets.get("schema")  if "schema"  in dbutils.widgets.getAll() else "worldcup"
table   = dbutils.widgets.get("table")   if "table"   in dbutils.widgets.getAll() else "bronze_raw_fifa_wc_2026_teams"

# This was a way to load and save the raw data in a landing zone and only update if it was updated on the remote
# abandoned in favor of pulling directly using kagglehub. but I might revisit
download_path = "/tmp/fifa_wc_2026"
uc_volume_path = "/Volumes/" + catalog + "/" + schema + "/landing"
bronze_table_path = "/Volumes/" + catalog + "/" + schema + "/landing/delta/" + table
version_path      = uc_volume_path + "/last_known_version.txt"

In [0]:
# ============================================================
# CELL 1 — get data from Kaggle and write to Bronze
# ============================================================
df_pandas = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "swaptr/fifa-wc-2026-teams",
    "teams.csv",
)
df_raw = spark.createDataFrame(df_pandas)

df_bronze = (
    df_raw
    .withColumn("_ingested_at",    F.current_timestamp())
    .withColumn("_ingestion_date", F.to_date(F.current_timestamp()))
)

print("Row count: " + str(df_bronze.count()))
df_bronze.printSchema()

spark.sql("CREATE SCHEMA IF NOT EXISTS " + catalog + "." + schema)

full_table_name = catalog + "." + schema + "." + table

# Write directly as a managed UC table instead of using LOCATION
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(full_table_name)
)

spark.sql("SELECT * FROM " + full_table_name + " LIMIT 5").show(truncate=False)